# Multiple Linear Regression

**Topic:** Supervised Learning — Regression

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, HTML
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how multiple linear regression extends the single-feature case to a matrix formulation
- **Explain** what multicollinearity is and why it distorts individual coefficient estimates
- **Interpret** a coefficient bar chart to identify which features the model weighted most heavily

> **Tip:** In the **Coefficient Shift Explorer** below, choose a feature to add to a two-feature baseline model and watch how much the existing coefficients move. The more correlated your choice is with an existing feature, the bigger the shift — that's multicollinearity in action.

---
## How we got here

This notebook builds directly on the single-variable case:

- **[supervised/01_linear_regression.ipynb](01_linear_regression.ipynb)** — the cost function, the normal equations, and the residual interpretation you need here
- **[math/linear_algebra/04_matrix_multiplication.ipynb](../math/linear_algebra/04_matrix_multiplication.ipynb)** — the dot product and matrix-vector multiply that turn many features into one prediction at once

The design matrix $X$ in the normal equations is the key new idea: stacking all your training examples as rows, each feature as a column, lets you solve for every coefficient simultaneously with one matrix formula.

---
## Why this matters for data science

Real-world prediction problems almost always involve more than one input. House prices depend on size, location, age, and condition — not just one number. Multiple linear regression extends the same interpretable framework to any number of features simultaneously.

It is also the mental model you need to understand regularization (Ridge and Lasso), logistic regression, and every other linear method. The coefficients are partial effects: each one tells you the change in y for a one-unit increase in that feature while all others stay fixed. That conditional interpretation is what makes regression coefficients so useful in scientific and business contexts.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Multiple linear regression sits at the **same top-left position** as simple linear regression: maximum interpretability, minimum complexity. Adding more features does not change the model family — it just adds more coefficients.

The practical complexity concern is multicollinearity. When two features are strongly correlated, the model cannot tell which one is responsible for changes in y, and both coefficients become unstable and uninterpretable. Regularization (next notebook) is the standard fix.

---
## How it learns

Everything from simple linear regression carries over: the model finds the coefficients that minimize MSE. The only change is that instead of fitting a line through a 2D scatter, it is fitting a **hyperplane** through a high-dimensional space.

Think of it as a flat table with as many legs as you have features. Each leg (coefficient) is adjusted up or down until the table surface passes as close as possible to all the data points floating in space. The table can tilt differently along each axis — that tilt is the coefficient for that feature.

The normal equations still give the exact solution in one step. But with many correlated features, the matrix $(X^\top X)$ can become nearly singular (close to non-invertible), which causes numerical instability and inflated variance in the coefficients. That is the multicollinearity problem, and it is why Ridge and Lasso exist.

---
## The math behind it

Multiple linear regression uses the same MSE loss as the simple case, but now $\boldsymbol{\theta}$ is a vector of $p{+}1$ coefficients:

$$\hat{\mathbf{y}} = X \boldsymbol{\theta}$$

$$J(\boldsymbol{\theta}) = \frac{1}{n} \| \hat{\mathbf{y}} - \mathbf{y} \|^2 = \frac{1}{n} \| X\boldsymbol{\theta} - \mathbf{y} \|^2$$

- $X$ — design matrix, shape $(n \times p{+}1)$; first column all ones (intercept)
- $\boldsymbol{\theta}$ — coefficient vector, shape $(p{+}1 \times 1)$
- $\mathbf{y}$ — target vector, shape $(n \times 1)$
- $\|\cdot\|^2$ — squared L2 norm (sum of squared elements)

**Normal equations** — exact closed-form minimum:

$$\boldsymbol{\theta}^* = (X^\top X)^{-1} X^\top \mathbf{y}$$

This is computationally feasible up to a few thousand features. For $p > 10{,}000$ or when $(X^\top X)$ is nearly singular, sklearn switches to an SVD-based pseudoinverse.

**Optimization criterion:** minimize MSE (equivalent to maximizing the likelihood under a Gaussian noise assumption).

---
## Categorical Variables in Regression

Linear regression requires all inputs to be numeric. A categorical feature — neighbourhood type, employment category, season — has to be one-hot encoded into k − 1 dummy variables (dropping one category as the reference) before it can enter the design matrix $X$ above.

Drop all k dummies instead of k − 1 and you get the **dummy variable trap**: the dummies sum to 1 for every row, which is perfectly collinear with the intercept column and makes $(X^\top X)$ singular.

```python
# pandas creates k-1 dummies automatically with drop_first=True
X_encoded = pd.get_dummies(df, columns=["neighbourhood"], drop_first=True)
```

See **[preprocessing/02_encoding_categorical_variables.ipynb](../preprocessing/02_encoding_categorical_variables.ipynb)** for the full range of encoding strategies.

---
## Assumptions in Multiple Regression

Multiple linear regression shares the same five assumptions as simple linear regression: linearity, independence, homoscedasticity, normality of residuals, and no multicollinearity. See **[01_linear_regression.ipynb](01_linear_regression.ipynb)** for the full treatment of each assumption.

Two assumptions deserve extra attention when you have multiple features.

**Multicollinearity becomes much more serious.** With one feature, multicollinearity is impossible. With many features, it is almost inevitable. When two features are correlated, their coefficients compensate for each other in ways that make individual slopes meaningless. The Variance Inflation Factor (VIF) in the next section is the standard tool for detecting this.

**Independence is harder to verify.** With many features, the residuals mix the effects of all predictors, and subtle autocorrelation patterns may only appear after accounting for each feature separately. The Durbin-Watson test in the statsmodels OLS summary is the practical starting point.

---
## Variance Inflation Factor (VIF)

VIF quantifies how much a coefficient's variance is inflated due to its correlation with other features. It answers: "If this feature were uncorrelated with all others, how much smaller would the variance of its coefficient be?"

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² obtained by regressing feature $j$ on all other features. If feature $j$ can be perfectly predicted from the others, VIF is infinite and $(X^\top X)$ is singular.

### Interpretation thresholds

| VIF | Severity | Action |
|-----|---------|--------|
| 1 | No correlation | No concern |
| 1-5 | Low to moderate | Usually acceptable |
| 5-10 | Moderate to high | Investigate; consider removing one feature |
| > 10 | Serious multicollinearity | Coefficient estimates are unreliable |

### What to do when VIF is high

- **Remove one of the correlated features.** If two features carry essentially the same information, keep the more interpretable one
- **Combine correlated features.** Create a composite score from the correlated pair
- **Use Ridge regression,** which shrinks coefficients toward zero and handles multicollinearity gracefully. See **[04_ridge_lasso_regression.ipynb](04_ridge_lasso_regression.ipynb)**

The table below shows VIF values for all eight California Housing features.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.datasets import fetch_california_housing
import pandas as pd

np.random.seed(42)
_hv = fetch_california_housing(as_frame=True)
_Xv = _hv.data

_vif = pd.DataFrame({
    "Feature": _Xv.columns,
    "VIF": [variance_inflation_factor(_Xv.values, i) for i in range(_Xv.shape[1])],
})
_vif["Severity"] = _vif["VIF"].apply(
    lambda v: "No concern" if v < 5 else ("Moderate" if v < 10 else "High"))
_vif = _vif.sort_values("VIF", ascending=False).reset_index(drop=True)

print("Variance Inflation Factor — California Housing Features")
print("=" * 54)
print(f"  {'Feature':<22} {'VIF':>8}   Severity")
print("-" * 54)
for _, row in _vif.iterrows():
    flag = " <<" if row["VIF"] > 5 else ""
    print(f"  {row['Feature']:<22} {row['VIF']:>8.2f}   {row['Severity']}{flag}")
print()
print("Features marked << exceed the VIF > 5 threshold")


---
## Reading OLS Output for Multiple Features

The statsmodels OLS summary is the same for multiple features as for simple regression, but three things deserve extra attention.

### Adjusted R² vs R²

**R²** always increases when you add any feature, even a random one. Never use R² alone to decide whether to add a feature.

**Adjusted R²** penalizes for the number of predictors and increases only when a new feature explains more variance than would be expected by random chance. When comparing models with different numbers of features, adjusted R² is the right criterion.

### Feature-level p-values

Each coefficient's p-value tests H\u2080: "this coefficient equals zero, holding all other features constant." Important nuances:

- Multicollinearity inflates standard errors and can make genuinely important features appear insignificant — always check VIF alongside p-values
- A p-value of 0.12 does not mean the feature has no effect; it means the evidence is weak given the sample size and noise level

### AIC and BIC for model comparison

| Criterion | Penalty | Use when |
|-----------|---------|---------|
| AIC | 2k | Weaker penalty; better for prediction |
| BIC | k·ln(n) | Stronger penalty; favors simpler models |

Lower values are better. Use AIC or BIC when comparing models with different feature sets fitted to the same data.

### Partial regression plots

A partial regression plot for feature j shows the relationship between feature j and the target after removing the effect of all other features. It is the "true" marginal relationship — the slope equals the multiple regression coefficient for feature j.

The code below fits the full OLS model and shows the partial regression plot for MedInc.

In [ ]:
import statsmodels.api as sm
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
_hm = fetch_california_housing(as_frame=True)
_Xm, _ym = _hm.data, _hm.target.values

_Xmc  = sm.add_constant(_Xm)
_mlr  = sm.OLS(_ym, _Xmc).fit()
print(_mlr.summary())
print()

# Partial regression plot for MedInc (feature index 0)
_Xmv   = _Xm.values
_others = np.delete(_Xmv, 0, axis=1)
_Xoc2   = sm.add_constant(_others)

_ey  = sm.OLS(_ym, _Xoc2).fit().resid
_exj = sm.OLS(_Xmv[:, 0], _Xoc2).fit().resid

_idxp = np.random.choice(len(_ey), 800, replace=False)
_ps, _pi2 = np.polyfit(_exj[_idxp], _ey[_idxp], 1)
_px = np.linspace(_exj[_idxp].min(), _exj[_idxp].max(), 100)

_fp = go.Figure()
_fp.add_trace(go.Scatter(x=_exj[_idxp], y=_ey[_idxp], mode="markers",
    marker=dict(color=PALETTE["primary"], size=4, opacity=0.5), name="Partial residuals"))
_fp.add_trace(go.Scatter(x=_px, y=_ps * _px + _pi2, mode="lines",
    line=dict(color=PALETTE["secondary"], width=2.5),
    name=f"Partial slope = {_ps:.3f}"))
_fp.update_layout(base_layout(
    "Partial Regression Plot — MedInc (controlling for all other features)",
    "MedInc residuals (after removing other features)",
    "y residuals (after removing other features)"))
_fp.show()


---
## Try it yourself

Four widgets below let you explore multicollinearity from different angles: coefficient shifts, the 3D geometry of a fitted plane, OLS diagnostics, and feature-level VIF/correlation/selection views.

### Coefficient Shift Explorer

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing
import numpy as np

np.random.seed(42)
_h_w1 = fetch_california_housing(as_frame=True)
_X_w1, _y_w1 = _h_w1.data, _h_w1.target.values
_base_feats_w1 = ["MedInc", "AveRooms"]
_addable_w1 = [f for f in _h_w1.feature_names if f not in _base_feats_w1]

out_multicol = Output()

add_feature_dd = Dropdown(
    options=_addable_w1,
    value="AveBedrms",
    description="Add feature:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="260px"),
)

def render_multicol(change=None):
    added = add_feature_dd.value

    # Baseline model: MedInc + AveRooms only
    Xb = _X_w1[_base_feats_w1].values
    Xbs = StandardScaler().fit_transform(Xb)
    lr_base = LinearRegression().fit(Xbs, _y_w1)

    # Extended model: baseline + the chosen feature
    feats_ext = _base_feats_w1 + [added]
    Xe = _X_w1[feats_ext].values
    Xes = StandardScaler().fit_transform(Xe)
    lr_ext = LinearRegression().fit(Xes, _y_w1)

    corr = _X_w1[_base_feats_w1 + [added]].corr()[added][_base_feats_w1]
    most_corr_feat = corr.abs().idxmax()
    most_corr_val = corr[most_corr_feat]
    base_idx = _base_feats_w1.index(most_corr_feat)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=_base_feats_w1, y=lr_base.coef_, name="Before adding feature",
        marker_color=PALETTE["muted"],
        text=[f"{c:+.3f}" for c in lr_base.coef_], textposition="outside",
    ))
    fig.add_trace(go.Bar(
        x=feats_ext, y=lr_ext.coef_, name=f"After adding {added}",
        marker_color=PALETTE["primary"],
        text=[f"{c:+.3f}" for c in lr_ext.coef_], textposition="outside",
    ))
    fig.update_layout(base_layout(
        title="Coefficient Shift When a Correlated Feature Is Added",
        xaxis_title="Feature", yaxis_title="Standardized coefficient",
    ))
    fig.update_layout(barmode="group")

    shift = lr_ext.coef_[base_idx] - lr_base.coef_[base_idx]
    caption = (
        f"<b>{added}</b> correlates most with <b>{most_corr_feat}</b> (r = {most_corr_val:+.2f}). "
        f"Adding it shifted the <b>{most_corr_feat}</b> coefficient by {shift:+.3f} "
        f"({lr_base.coef_[base_idx]:+.3f} → {lr_ext.coef_[base_idx]:+.3f}). "
        f"The stronger the correlation, the more unstable that shift becomes."
    )

    with out_multicol:
        clear_output(wait=True)
        fig.show()
        display(HTML(
            f'<div style="font-family:{FONT["family"]};font-size:13px;'
            f'background:#f8f9fa;padding:10px 14px;border-radius:6px;margin-top:6px;">{caption}</div>'
        ))

add_feature_dd.observe(render_multicol, names="value")
display(VBox([add_feature_dd, out_multicol]))
render_multicol()


### 3D Regression Plane

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing
import numpy as np

np.random.seed(42)
_h_w2 = fetch_california_housing(as_frame=True)
_X_w2, _y_w2 = _h_w2.data, _h_w2.target.values
_idx_w2 = np.random.RandomState(42).choice(len(_y_w2), 300, replace=False)
_all_features_w2 = list(_h_w2.feature_names)

out_plane = Output()

feat_x_dd = Dropdown(options=_all_features_w2, value="MedInc",
    description="X feature:", style={"description_width": "90px"},
    layout=widgets.Layout(width="280px"))
feat_z_dd = Dropdown(options=_all_features_w2, value="AveRooms",
    description="Z feature:", style={"description_width": "90px"},
    layout=widgets.Layout(width="280px"))

def render_plane(change=None):
    fx, fz = feat_x_dd.value, feat_z_dd.value
    if fx == fz:
        with out_plane:
            clear_output(wait=True)
            print("Choose two different features.")
        return
    xv = _X_w2[fx].values[_idx_w2]
    zv = _X_w2[fz].values[_idx_w2]
    yv = _y_w2[_idx_w2]
    lr2 = LinearRegression().fit(np.column_stack([xv, zv]), yv)
    y_pred = lr2.predict(np.column_stack([xv, zv]))
    r2 = r2_score(yv, y_pred)

    xg = np.linspace(xv.min(), xv.max(), 15)
    zg = np.linspace(zv.min(), zv.max(), 15)
    Xg, Zg = np.meshgrid(xg, zg)
    Yg = lr2.intercept_ + lr2.coef_[0] * Xg + lr2.coef_[1] * Zg

    residual_traces = [
        go.Scatter3d(x=[xv[i], xv[i]], y=[zv[i], zv[i]], z=[yv[i], y_pred[i]],
            mode="lines", line=dict(color=PALETTE["muted"], width=1),
            showlegend=False, hoverinfo="skip")
        for i in range(len(xv))
    ]

    fig = go.Figure(data=[
        go.Surface(x=Xg, y=Zg, z=Yg,
            colorscale=[[0, PALETTE["primary"]], [1, PALETTE["primary"]]],
            opacity=0.5, showscale=False, name="Fitted plane"),
        go.Scatter3d(x=xv, y=zv, z=yv, mode="markers",
            marker=dict(color=PALETTE["secondary"], size=3, opacity=0.7), name="Districts"),
        *residual_traces,
    ])
    fig.update_layout(
        title=dict(text=f"Regression Plane — {fx} & {fz}  |  R²={r2:.3f}",
                    font=dict(size=FONT["size_title"], family=FONT["family"])),
        scene=dict(xaxis_title=fx, yaxis_title=fz, zaxis_title="Median House Value"),
        height=520, margin=dict(l=0, r=0, t=50, b=0),
        paper_bgcolor=PALETTE["background"],
    )
    with out_plane:
        clear_output(wait=True)
        fig.show()

feat_x_dd.observe(render_plane, names="value")
feat_z_dd.observe(render_plane, names="value")
display(VBox([HBox([feat_x_dd, feat_z_dd]), out_plane]))
render_plane()


### OLS Diagnostics Explorer

In [ ]:
import statsmodels.api as sm
from sklearn.datasets import fetch_california_housing

_h_w3 = fetch_california_housing(as_frame=True)
_Xc_w3 = sm.add_constant(_h_w3.data)
_ols_w3 = sm.OLS(_h_w3.target.values, _Xc_w3).fit()

out_ols = Output()

ols_toggle = widgets.ToggleButtons(
    options=["R² explanation", "Coefficient p-values", "Diagnostic tests"],
    value="R² explanation",
)

def _explanation(view):
    if view == "R² explanation":
        return (
            f"<b>R² = {_ols_w3.rsquared:.3f}</b>, Adjusted R² = {_ols_w3.rsquared_adj:.3f}. "
            f"The model explains {_ols_w3.rsquared*100:.1f}% of the variance in house value. "
            f"F-statistic = {_ols_w3.fvalue:,.1f} "
            f"(p {'&lt; 0.001' if _ols_w3.f_pvalue < 0.001 else f'= {_ols_w3.f_pvalue:.4f}'}) "
            f"— the model as a whole is statistically significant."
        )
    if view == "Coefficient p-values":
        return (
            "Each bar below is a 95% confidence interval for one coefficient. "
            "Green means p &lt; 0.05 (significant), orange means 0.05–0.10 (marginal), "
            "red means p &gt; 0.10 (not significant). If a bar crosses zero, the coefficient "
            "is not significantly different from zero."
        )
    dw = sm.stats.stattools.durbin_watson(_ols_w3.resid)
    # Condition number on raw feature scales conflates units with collinearity
    # (Population is ~10,000x the scale of Latitude), so this diagnostic is
    # computed on standardized features instead for a scale-invariant reading.
    _Xs_check = (_h_w3.data - _h_w3.data.mean()) / _h_w3.data.std()
    _cond_std = np.linalg.cond(sm.add_constant(_Xs_check).values)
    return (
        f"Durbin-Watson = {dw:.3f} (near 2.0 = no autocorrelation). "
        f"Condition number (standardized) = {_cond_std:,.0f} "
        f"({'flagging possible multicollinearity' if _cond_std > 30 else 'no strong multicollinearity signal'})."
    )

def render_ols(change=None):
    view = ols_toggle.value
    with out_ols:
        clear_output(wait=True)
        display(HTML(
            f'<div style="font-family:{FONT["family"]};font-size:13px;'
            f'background:#f8f9fa;padding:12px 16px;border-radius:6px;">'
            f'{_explanation(view)}</div>'
        ))
        if view == "Coefficient p-values":
            names = _ols_w3.params.index.tolist()
            coefs = _ols_w3.params.values
            ci = _ols_w3.conf_int()
            pvals = _ols_w3.pvalues.values
            colors = [PALETTE["accent"] if p < 0.05 else ("#F59F00" if p < 0.10 else PALETTE["secondary"]) for p in pvals]
            err_low = coefs - ci[0].values
            err_high = ci[1].values - coefs
            fig = go.Figure(data=go.Bar(
                x=coefs, y=names, orientation="h",
                marker_color=colors,
                error_x=dict(type="data", symmetric=False, array=err_high, arrayminus=err_low),
            ), layout=base_layout(
                title="Coefficient Forest Plot — 95% Confidence Intervals",
                xaxis_title="Coefficient value", yaxis_title="",
            ))
            fig.add_vline(x=0, line_color=PALETTE["muted"], line_width=1.5, line_dash="dash")
            fig.show()

ols_toggle.observe(render_ols, names="value")
display(VBox([ols_toggle, out_ols]))
render_ols()


### Feature Diagnostics Explorer

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from sklearn.datasets import fetch_california_housing
import numpy as np

_h_w4 = fetch_california_housing(as_frame=True)
_X_w4, _y_w4 = _h_w4.data, _h_w4.target.values
_feat_w4 = list(_h_w4.feature_names)

out_selection = Output()

view_dd = Dropdown(
    options=["VIF Table", "Correlation Heatmap", "Feature Selection (forward)"],
    value="VIF Table",
    description="View:", style={"description_width": "60px"},
    layout=widgets.Layout(width="320px"),
)

def _vif_data():
    return [variance_inflation_factor(_X_w4.values, i) for i in range(_X_w4.shape[1])]

def _vif_fig(vif_vals):
    colors = [PALETTE["accent"] if v < 5 else ("#F59F00" if v < 10 else PALETTE["secondary"]) for v in vif_vals]
    fig = go.Figure(data=go.Bar(
        x=_feat_w4, y=vif_vals, marker_color=colors,
        text=[f"{v:.1f}" for v in vif_vals], textposition="outside",
    ), layout=base_layout("Variance Inflation Factor by Feature", "Feature", "VIF"))
    fig.add_hline(y=5, line_color="#F59F00", line_dash="dash", annotation_text="VIF=5")
    fig.add_hline(y=10, line_color=PALETTE["secondary"], line_dash="dash", annotation_text="VIF=10")
    return fig

def _vif_caption(vif_vals):
    flagged = [f for f, v in zip(_feat_w4, vif_vals) if v > 5]
    if flagged:
        return (f"Features above the VIF=5 line: <b>{', '.join(flagged)}</b>. "
                f"Their coefficients should be interpreted cautiously, since each is well predicted "
                f"by the others.")
    return "No feature exceeds VIF=5 here — coefficients can be interpreted individually with reasonable confidence."

def _corr_fig():
    corr = _X_w4.corr()
    z = corr.values
    fig = go.Figure(data=go.Heatmap(
        z=z, x=_feat_w4, y=_feat_w4, colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
        text=np.round(z, 2), texttemplate="%{text}",
    ), layout=base_layout("Feature Correlation Heatmap", "", ""))
    fig.update_layout(height=420)
    return fig

def _corr_caption():
    corr = _X_w4.corr()
    vals = corr.where(~np.eye(len(corr), dtype=bool))
    max_pair = vals.abs().stack().idxmax()
    max_val = corr.loc[max_pair]
    return (f"Strongest correlation: <b>{max_pair[0]}</b> and <b>{max_pair[1]}</b> "
            f"(r = {max_val:+.2f}). Pairs above about |r| = 0.7 are the ones most likely to "
            f"produce the unstable coefficients seen in the VIF table.")

def _forward_selection_data():
    remaining = list(_feat_w4)
    selected = []
    adj_r2_path = []
    labels = []
    while remaining:
        best_feat, best_adj_r2 = None, -np.inf
        for f in remaining:
            cols = selected + [f]
            Xc = sm.add_constant(_X_w4[cols])
            model = sm.OLS(_y_w4, Xc).fit()
            if model.rsquared_adj > best_adj_r2:
                best_adj_r2, best_feat = model.rsquared_adj, f
        selected.append(best_feat)
        remaining.remove(best_feat)
        adj_r2_path.append(best_adj_r2)
        labels.append(best_feat)
    return labels, adj_r2_path

def _forward_selection_fig(labels, adj_r2_path):
    fig = go.Figure(data=go.Scatter(
        x=list(range(1, len(labels) + 1)), y=adj_r2_path, mode="lines+markers+text",
        text=labels, textposition="top center",
        line=dict(color=PALETTE["primary"], width=2.5), marker=dict(size=8),
    ), layout=base_layout(
        "Forward Selection — Adjusted R² as Features Are Added",
        "Step (feature added)", "Adjusted R²",
    ))
    fig.update_layout(xaxis=dict(tickmode="linear", dtick=1))
    return fig

def _forward_caption(labels, adj_r2_path):
    gains = [adj_r2_path[0]] + [adj_r2_path[i] - adj_r2_path[i - 1] for i in range(1, len(adj_r2_path))]
    return (f"<b>{labels[0]}</b> is added first and gives the single biggest jump in adjusted R² "
            f"(+{gains[0]:.3f}). By the time <b>{labels[-1]}</b> is added last, it only contributes "
            f"{gains[-1]:+.3f} — diminishing returns as correlated features start competing for "
            f"the same signal.")

def render_selection(change=None):
    view = view_dd.value
    with out_selection:
        clear_output(wait=True)
        if view == "VIF Table":
            vif_vals = _vif_data()
            _vif_fig(vif_vals).show()
            caption = _vif_caption(vif_vals)
        elif view == "Correlation Heatmap":
            _corr_fig().show()
            caption = _corr_caption()
        else:
            labels, path = _forward_selection_data()
            _forward_selection_fig(labels, path).show()
            caption = _forward_caption(labels, path)
        display(HTML(
            f'<div style="font-family:{FONT["family"]};font-size:13px;'
            f'background:#f8f9fa;padding:10px 14px;border-radius:6px;margin-top:6px;">{caption}</div>'
        ))

view_dd.observe(render_selection, names="value")
display(VBox([view_dd, out_selection]))
render_selection()


---
## What's happening?

**Coefficient Shift Explorer:** adding a feature that's correlated with one already in the model shifts the existing coefficients — sometimes a lot. This is multicollinearity in action: the model can't cleanly separate the two overlapping effects, so it splits credit between them in a way that depends on exactly which features happen to be in the model. The bigger the correlation between the added feature and an existing one, the bigger the shift you'll see.

**3D Regression Plane:** this shows why fitting a joint model beats repeating simple regression per feature. The plane captures both features' effects at once, accounting for their correlation, and the residuals from the plane are typically smaller than fitting either feature alone.

**OLS Diagnostics Explorer:** the R² and p-value views tell you whether the model, and each coefficient, is statistically meaningful. But cross-reference the p-values against VIF — a feature can look statistically insignificant here purely because multicollinearity has inflated its standard error, not because it actually has no effect.

**Feature Diagnostics Explorer:** the VIF table and correlation heatmap are two views of the same underlying problem — the heatmap shows you which pairs are correlated, the VIF table shows you how much that correlation is inflating each coefficient's variance. Forward selection shows a related but different idea: as correlated features get added, each new one contributes less, since it's largely re-explaining variance the model already captured.

The throughline across all four: coefficients in a multiple regression model are **conditional** effects. Change what else is in the model and the coefficients change. That is not a bug — it is what partial effects mean.

---
## Feature Selection in Multiple Regression

When you have many potential features, deciding which to include is itself a modeling decision. Traditional approaches — forward selection, backward elimination, stepwise combinations — search over models by p-value or AIC, but they've fallen out of favor: they produce overly optimistic R² values and p-values that aren't statistically valid after the search, since the procedure implicitly tests exponentially many models.

**Modern alternative:** Lasso regression shrinks irrelevant coefficients to exactly zero as part of fitting, avoiding the multiple-testing problem. See **[04_ridge_lasso_regression.ipynb](04_ridge_lasso_regression.ipynb)** and **[feature_engineering/07_feature_selection.ipynb](../feature_engineering/07_feature_selection.ipynb)** for the full treatment — the Feature Diagnostics Explorer above previews what forward selection looks like in practice.

---
## Interaction Terms

An interaction term captures the idea that one feature's effect depends on the value of another — without it, the model assumes every feature contributes independently and additively.

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_{12} x_1 x_2$$

```python
df["hours_x_support"] = df["hours"] * df["support"]
```

$\theta_{12}$ represents how much the slope of $x_1$ changes per one-unit increase in $x_2$ — so $x_1$'s effect becomes $\theta_1 + \theta_{12} x_2$, which depends on $x_2$'s value. Interactions add multicollinearity risk, so always center your features first. See **[feature_engineering/04_interaction_features.ipynb](../feature_engineering/04_interaction_features.ipynb)** for examples and visualization strategies.

---
## Key hyperparameters

**`fit_intercept`** (default `True`) — whether to add an intercept term. Almost always leave this `True` unless you have a domain-specific reason to force the hyperplane through the origin.

**`positive`** (default `False`) — constrains all coefficients to be non-negative. Useful in mixture models where negative contributions make no physical sense.

**`n_jobs`** (default `None`) — number of parallel jobs for the computation. Set to `-1` to use all CPU cores on very large feature matrices.

For the full list of hyperparameters, see the sklearn documentation:
[https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Coefficients show direction and magnitude of each feature's effect | Assumes linear relationships between every feature and the target |
| Scales to hundreds of features with negligible extra compute | Multicollinearity destabilizes individual coefficients |
| Exact closed-form solution — no tuning or learning rate required | Sensitive to outliers (they distort the hyperplane) |
| Feature importance is directly readable from coefficient magnitudes | All features get nonzero coefficients even irrelevant ones |
| Strong statistical framework (confidence intervals, p-values available) | Poor extrapolation outside the training data range |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| You have multiple features and want to understand each one's contribution | Features are highly correlated with each other (use Ridge or Lasso) |
| Interpretability and transparency are required | The target has a nonlinear relationship with features |
| You need a fast, scalable baseline for a regression problem | You have far more features than training examples (p >> n) |
| Each feature contributes additively and independently | You need automatic feature selection (Lasso is better for that) |
| You want statistical inference: p-values and confidence intervals | You suspect interaction effects between features |

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import numpy as np, plotly.graph_objects as go

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

train_r2 = r2_score(y_train, model.predict(X_train))
test_r2 = r2_score(y_test, model.predict(X_test))
test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²:  {test_r2:.4f}  (gap: {train_r2 - test_r2:+.4f})")
print(f"Test RMSE: {test_rmse:.4f}")
print()
print("A small train/test gap here is expected: linear regression has low variance,")
print("so it rarely overfits even without regularization.")

coefs = model.coef_
colors = [PALETTE["primary"] if c > 0 else PALETTE["secondary"] for c in coefs]

fig = go.Figure(data=[
    go.Bar(
        x=housing.feature_names,
        y=coefs,
        marker_color=colors,
        marker_line=dict(width=0),
        text=[f"{c:+.3f}" for c in coefs],
        textposition="outside",
    )
], layout=base_layout(
    title="Standardized Coefficients — Fitted on Training Data",
    xaxis_title="Feature",
    yaxis_title="Coefficient (standardized units)",
))
fig.update_layout(showlegend=False)
fig.show()


---
## Real-world example: All eight features predicting house value

The chart above uses all eight California Housing features with standardized inputs (so coefficient sizes are directly comparable) and is fitted on the training split only, so its coefficients are the same ones used to generate the test-set R² and RMSE printed above.

- **Notice:** MedInc has the largest positive coefficient by a wide margin — median income is the dominant signal even after controlling for all other features
- **Notice:** Latitude and Longitude both carry signal, reflecting that geography (proximity to coast and job centers) affects prices after controlling for income and size
- **Notice:** AveBedrms has a negative coefficient — more bedrooms per household can indicate overcrowding, not luxury, and the model has learned this nuance
- **Notice:** the train/test R² gap is small — linear regression's low variance means it rarely overfits, even without regularization

> **Discussion question:** AveOccup (average occupancy per household) has a negative coefficient. Can you explain in plain English what this means economically?

### Where multiple linear regression is used in practice

| Industry | Application | Why multiple features help |
|---|---|---|
| Finance | Risk modeling | Many macroeconomic factors jointly predict default |
| Healthcare | Clinical outcome prediction | Patient demographics, labs, and history all contribute |
| Marketing | Marketing mix modeling | Spend across many channels affects revenue jointly |
| Retail | Demand forecasting | Price, promotions, seasonality, and competition all matter |
| HR analytics | Salary benchmarking | Role, experience, location, and skills interact |

> **Multiple linear regression extends the single-feature line to a hyperplane in any number of dimensions — each coefficient tells you how much the target changes for a one-unit increase in that feature, holding all others constant.**

---
*Next up: 03 — Polynomial Regression, adding curves by transforming features before fitting*